# fMRIPrep — Slurm Job Generation

This notebook generates per-subject Slurm job scripts for fMRIPrep and prints the `sbatch` commands to submit them to the HPC cluster.

**This project uses no-fieldmap correction** (`--ignore fieldmaps`). See `README_3_fmriprep.md` for a description of available SDC modes and how to switch.

**Before running:**
- BIDS conversion (Step 2) must be complete
- `scripts/FMRIPREP/home_dir/` must exist (used as a writable home for fMRIPrep v25+)

#### History
- 1/22/20 matt, jeesung, nicole — initial SLURM setup
- 5/22/20 mbod — updated for MURI fmriprep pipeline
- 7/9/21 dcosme — revised for bbprime
- Refactored for CNLab pipeline documentation

## 1. Imports and Path Setup

Paths are derived from this notebook's location (`scripts/FMRIPREP/`) so the notebook runs without editing as long as the expected BIDS data is in place.

In [ ]:
import os
import glob

# ── Derive project root from this notebook's location ─────────────────────────
# Assumes notebook is at: {project}/scripts/FMRIPREP/
project_dir  = os.path.abspath('../../')
bids_dir     = os.path.join(project_dir, 'data/bids_data')

# ── fMRIPrep output directories ───────────────────────────────────────────────
# This project uses no-correction derivatives (no SDC applied)
derivatives_dir = os.path.join(bids_dir, 'derivatives')
working_dir     = os.path.join(derivatives_dir, 'working')
home_dir        = os.path.join(project_dir, 'scripts/FMRIPREP/home_dir')  # writable home for fMRIPrep v25+

# ── Job script location ───────────────────────────────────────────────────────
slurm_dir = os.path.join(project_dir, 'scripts/FMRIPREP/jobs')

# ── Server-side tool paths (not project-specific) ─────────────────────────────
freesurfer_license = '/data00/tools/freesurfer/license.txt'
fmriprep_sif       = '/data00/tools/singularity_images/fmriprep-20.0.6.simg'

# ── Create directories ────────────────────────────────────────────────────────
for d in [derivatives_dir, working_dir, home_dir, slurm_dir, os.path.join(slurm_dir, 'out')]:
    os.makedirs(d, exist_ok=True)

print(f'Project root : {project_dir}')
print(f'BIDS dir     : {bids_dir}')
print(f'Derivatives  : {derivatives_dir}')
print(f'Jobs dir     : {slurm_dir}')
print(f'\nBIDS dir exists        : {os.path.exists(bids_dir)}')
print(f'FreeSurfer license     : {os.path.exists(freesurfer_license)}')
print(f'fMRIPrep image exists  : {os.path.exists(fmriprep_sif)}')

## 2. Detect Subjects

Subjects are auto-detected from the BIDS directory. The cell below also shows which subjects already have job files, so you can avoid re-generating them unnecessarily.

In [ ]:
# All subjects in bids_data/
all_subs = sorted([
    os.path.basename(d)
    for d in glob.glob(os.path.join(bids_dir, 'sub-*'))
    if os.path.isdir(d)
])

# Subjects that already have a job file
existing_jobs = [
    f.replace('fmriprep-nocorrection_', '').replace('.job', '')
    for f in os.listdir(slurm_dir)
    if f.endswith('.job')
]

# Subjects that still need a job file
subs_to_process = [s for s in all_subs if s not in existing_jobs]

print(f'Total subjects in BIDS : {len(all_subs)}')
print(f'Already have job files : {len(existing_jobs)}')
print(f'To generate            : {len(subs_to_process)}')
print(f'\nSubjects to process: {subs_to_process}')

## 3. Job Template

This template uses **no fieldmap correction** (`--ignore fieldmaps`), which is the mode used for this project.

**To switch modes**, replace `--ignore slicetiming fieldmaps` with one of:
- *(remove the flag entirely)* — use B0 fieldmaps from BIDS `fmap/` directory
- `--use-syn-sdc --ignore fieldmaps` — fieldmap-less SyN correction from T1w

See `README_3_fmriprep.md` for a full comparison of modes.

In [ ]:
job_template = r'''#!/bin/bash
#SBATCH --job-name=fmriprep-nocorrection_{ID}
#SBATCH --output=out/fmriprep-nocorrection_{ID}.out
#SBATCH --error=out/fmriprep-nocorrection_{ID}.err
#SBATCH --time=2-00:00:00
#SBATCH --cpus-per-task=8
#SBATCH --mem=15GB

srun singularity run --cleanenv \
    -B {freesurfer_license}:/opt/freesurfer/license.txt \
    -B {bids_dir}:/data \
    -B {derivatives_dir}:/out \
    -B {working_dir}:/work \
    -B {home_dir}:/home/fmriprep \
    --home /home/fmriprep \
    {fmriprep_sif} /data /out participant \
    --participant-label {ID} \
    -w /work \
    --ignore slicetiming fieldmaps \
    --nthreads 8 \
    --skip-bids-validation
'''

# Preview what one job file looks like
preview = job_template.format(
    ID='sub-001',
    freesurfer_license=freesurfer_license,
    bids_dir=bids_dir,
    derivatives_dir=derivatives_dir,
    working_dir=working_dir,
    home_dir=home_dir,
    fmriprep_sif=fmriprep_sif
)
print(preview)

## 4. Generate Job Scripts

In [ ]:
for sub in subs_to_process:
    # Strip 'sub-' prefix — fMRIPrep's --participant-label expects bare ID
    sub_id = sub.replace('sub-', '')

    job_content = job_template.format(
        ID=sub_id,
        freesurfer_license=freesurfer_license,
        bids_dir=bids_dir,
        derivatives_dir=derivatives_dir,
        working_dir=working_dir,
        home_dir=home_dir,
        fmriprep_sif=fmriprep_sif
    )

    job_path = os.path.join(slurm_dir, f'fmriprep-nocorrection_{sub}.job')
    with open(job_path, 'w') as f:
        f.write(job_content)

    print(f'Created: {job_path}')

print(f'\n{len(subs_to_process)} job scripts written to: {slurm_dir}')

## 5. Print sbatch Commands

SSH to the Slurm master node and paste the output below:
```bash
ssh <username>@asc.upenn.edu@cls000
```

In [ ]:
print(f'cd {slurm_dir}\n')
for sub in subs_to_process:
    print(f'sbatch -D {slurm_dir} -c 8 fmriprep-nocorrection_{sub}.job')

---
## 6. Re-run a Subject (Delete Old Output First)

If a subject's job failed or you need to re-run with different settings, delete the old derivative and working files before resubmitting to prevent stale-file conflicts.

In [ ]:
# Set the subject ID to re-run (with 'sub-' prefix)
rerun_sub = 'sub-001'

deriv_path   = os.path.join(derivatives_dir, rerun_sub)
freesurfer_path = os.path.join(derivatives_dir, 'freesurfer', rerun_sub)

# Show what will be deleted — comment out the rm lines until you are sure
print(f'Will remove:')
print(f'  {deriv_path}')
print(f'  {freesurfer_path}')
print(f'  Working directory nodes matching: *{rerun_sub}*_wf')
print()
print('Uncomment the lines below to actually delete:')

# !rm -rf {deriv_path}
# !rm -rf {freesurfer_path}
# !find {working_dir} -maxdepth 2 -name "*{rerun_sub.replace('sub-', '')}*_wf" -exec rm -rf {{}} +